# LeetCode #1079: Letter Tile Possibilities

https://leetcode.com/problems/letter-tile-possibilities/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n! \cdot n)$ | $O(n)$ |
| **Optimal: Frequency Backtracking ★** | $O(n!)$ | $O(26)$ |

---

## Understanding the Methods

### Brute Force
Generate every permutation of `tiles`, collect them in a set to deduplicate, and count. Blows up quickly for repeated characters and $n \le 7$.

### Optimal: Frequency Backtracking ★
Track the frequency of each of the 26 letters. At each step, try placing any letter whose count is positive, increment the answer, decrement the count, recurse, then restore. This naturally avoids duplicates because we iterate over distinct letters rather than positions.

**Constraints:**
* $1 \le tiles.length \le 7$
* `tiles` consists of uppercase English letters

## Solutions

### C#

In [ ]:
public class Solution {
    public int NumTilePossibilities(string tiles) {
        int[] freq = new int[26];
        foreach (char c in tiles) freq[c - 'A']++;
        return Backtrack(freq);
    }

    int Backtrack(int[] freq) {
        int count = 0;
        for (int i = 0; i < 26; i++) {
            if (freq[i] == 0) continue;
            // Choose letter i as the next tile — every choice is a valid sequence
            count++;
            freq[i]--;
            count += Backtrack(freq); // sequences built on top of this prefix
            freq[i]++;               // restore for next choice
        }
        return count;
    }
}

### Python

In [ ]:
class Solution:
    def numTilePossibilities(self, tiles: str) -> int:
        freq = [0] * 26
        for ch in tiles:
            freq[ord(ch) - ord('A')] += 1

        def backtrack() -> int:
            total = 0
            for i in range(26):
                if freq[i] == 0:
                    continue
                # Placing this letter creates one new valid sequence
                total += 1
                freq[i] -= 1
                total += backtrack()  # count all extensions of this prefix
                freq[i] += 1         # backtrack to try other letters
            return total

        return backtrack()

### Go

In [ ]:
func numTilePossibilities(tiles string) int {
	freq := [26]int{}
	for _, ch := range tiles {
		freq[ch-'A']++
	}

	var backtrack func() int
	backtrack = func() int {
		total := 0
		for i := 0; i < 26; i++ {
			if freq[i] == 0 {
				continue
			}
			// Each distinct letter we pick creates a new valid sequence
			total++
			freq[i]--
			total += backtrack() // count extensions
			freq[i]++            // restore
		}
		return total
	}
	return backtrack()
}

### Rust

In [ ]:
impl Solution {
    pub fn num_tile_possibilities(tiles: String) -> i32 {
        let mut freq = [0i32; 26];
        for ch in tiles.bytes() {
            freq[(ch - b'A') as usize] += 1;
        }
        Self::backtrack(&mut freq)
    }

    fn backtrack(freq: &mut [i32; 26]) -> i32 {
        let mut total = 0;
        for i in 0..26 {
            if freq[i] == 0 { continue; }
            // Placing letter i is itself a valid new sequence
            total += 1;
            freq[i] -= 1;
            total += Self::backtrack(freq); // count all continuations
            freq[i] += 1;                  // restore
        }
        total
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `tiles="AAB"`
Distinct sequences: A, B, AA, AB, BA, AAB, ABA, BAA → **8**.
The frequency array starts as A:2, B:1. The backtracker naturally avoids generating duplicate "A" picks because it works on the count.

### 2. Slightly Complex
**Input:** `tiles="AAABBC"`
With 3 A's, 2 B's, and 1 C the search tree is wider but backtracking prunes duplicate subtrees automatically. Expected answer: **188**.

### 3. Edge Case: Time Factor
**Input:** `tiles="ABCDEFG"` (all distinct, 7 letters)
Every permutation is unique, so the count equals $\sum_{k=1}^{7} \frac{7!}{(7-k)!} = 13700$. Backtracking visits each of the $O(n!)$ nodes once.

### 4. Edge Case: Space Factor
**Input:** `tiles="AAAAAAA"` (7 identical letters)
Only 7 distinct sequences exist (lengths 1–7). The recursion stack goes 7 levels deep but uses $O(26) = O(1)$ space in the frequency array.

### 5. Almost-Impossible but Plausible
**Input:** `tiles="AABBCC"` (balanced duplicates, near-max length)
Frequency backtracking avoids the $6! = 720$ brute-force permutations and counts only distinct arrangements: **96**. Without deduplication the brute-force set would still correctly output 96, but would evaluate up to 720 candidates.